# 🤖 Análisis Comparativo de Algoritmos de IA
## Motor Predictivo del Gemelo Digital - Minería

**Metodología:** CRISP-DM (6 fases)  
**Algoritmos evaluados:** Random Forest, XGBoost, SVM, CNN-LSTM, LSTM-AE+RF  
**Objetivo:** Predicción de fallas en motores de equipos de carguío minero

---
## ⚙️ Configuración Inicial

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Agregar ruta del proyecto
PROJECT_DIR = os.path.dirname(os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, PROJECT_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuración de gráficos
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print('✅ Librerías importadas correctamente')

---
## 🚀 Inicializar Motor de IA

In [ ]:
from modules.motor_ia import MotorPredictivo

# Inicializar motor
motor = MotorPredictivo()
print('✅ MotorPredictivo inicializado')

---
## 📊 FASE 2: Comprensión de los Datos (EDA)

### Cargar Datos

In [ ]:
# Cargar datos desde la base de datos
motor.cargar_datos()
df = motor.datos

print(f'📊 Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'📅 Rango temporal: {df.fecha_hora.min()} → {df.fecha_hora.max()}')
print(f'🚛 Equipos únicos: {df.equipo_id.nunique()}')

In [ ]:
# Vista previa de los datos
variables_sensores = [
    'temp_motor', 'presion_aceite', 'rpm_motor', 'horas_motor',
    'presion_hidraulica', 'temp_aceite_hidraulico', 'nivel_aceite_hidraulico',
    'desgaste_pastillas', 'presion_neumaticos', 'desgaste_neumaticos',
    'nivel_combustible', 'consumo_combustible', 'carga_actual'
]

df[variables_sensores + ['equipo_id', 'fecha_hora']].head()

### Estadísticas Descriptivas

In [ ]:
# Estadísticas descriptivas
df[variables_sensores].describe().round(2)

### Análisis de Valores Nulos

In [ ]:
# Análisis de valores nulos
nulos = df[variables_sensores].isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)

nulos_df = pd.DataFrame({'Valores Nulos': nulos, 'Porcentaje (%)': pct_nulos})
nulos_df = nulos_df[nulos_df['Valores Nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)

if len(nulos_df) > 0:
    print('⚠️ Variables con valores nulos:')
    display(nulos_df)
else:
    print('✅ No hay valores nulos en las variables de sensores')

### Distribuciones de Variables

In [ ]:
# Histogramas de variables principales
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

vars_plot = ['temp_motor', 'presion_aceite', 'rpm_motor', 
            'presion_hidraulica', 'desgaste_pastillas', 'desgaste_neumaticos',
            'nivel_combustible', 'consumo_combustible', 'carga_actual']

for i, var in enumerate(vars_plot):
    sns.histplot(df[var], kde=True, ax=axes[i], bins=30)
    axes[i].set_title(f'Distribución: {var}')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

### Matriz de Correlaciones

In [ ]:
# Matriz de correlaciones
corr_matrix = df[variables_sensores].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Matriz de Correlaciones - Variables de Sensores', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

### Series Temporales

In [ ]:
# Series temporales de parámetros críticos
df['fecha_hora'] = pd.to_datetime(df['fecha_hora'])
df_sorted = df.sort_values('fecha_hora')

fig = make_subplots(rows=2, cols=2, 
                    subplot_titles=('Temperatura Motor (°C)', 'Presión Aceite (PSI)', 
                                   'RPM Motor', 'Desgaste Neumáticos (%)'))

fig.add_trace(go.Scatter(x=df_sorted['fecha_hora'], y=df_sorted['temp_motor'], 
                         mode='lines', name='Temp. Motor', line=dict(color='#e74c3c')),
              row=1, col=1)

fig.add_trace(go.Scatter(x=df_sorted['fecha_hora'], y=df_sorted['presion_aceite'], 
                         mode='lines', name='Pres. Aceite', line=dict(color='#3498db')),
              row=1, col=2)

fig.add_trace(go.Scatter(x=df_sorted['fecha_hora'], y=df_sorted['rpm_motor'], 
                         mode='lines', name='RPM', line=dict(color='#2ecc71')),
              row=2, col=1)

fig.add_trace(go.Scatter(x=df_sorted['fecha_hora'], y=df_sorted['desgaste_neumaticos'], 
                         mode='lines', name='Desgaste', line=dict(color='#f39c12')),
              row=2, col=2)

fig.update_layout(height=800, showlegend=False, title_text='📈 Series Temporales de Parámetros Críticos')
fig.show()

### Análisis Exploratorio Completo (EDA)

In [ ]:
# Ejecutar EDA completo del motor
eda_resultados = motor.analisis_exploratorio()

print('📊 Resumen EDA:')
print(f'   • Registros totales: {eda_resultados["num_registros"]}')
print(f'   • Variables analizadas: {len(eda_resultados["variables_sensores"])}')
print(f'   • Outliers detectados: {sum(eda_resultados["outliers_por_variable"].values())}')

if eda_resultados.get('desbalance_clases'):
    print(f'   • Desbalance de clases: {eda_resultados["desbalance_clases"]}')

---
## ⚙️ FASE 3: Preparación de los Datos

In [ ]:
# Preparar datos para modelado
motor.preparar_datos()

print('✅ Datos preparados exitosamente')
print()
print('📊 División de datos:')
print(f'   • Entrenamiento: {len(motor.X_train)} muestras ({len(motor.X_train)/len(motor.datos)*100:.0f}%)')
print(f'   • Validación:    {len(motor.X_val)} muestras ({len(motor.X_val)/len(motor.datos)*100:.0f}%)')
print(f'   • Prueba:        {len(motor.X_test)} muestras ({len(motor.X_test)/len(motor.datos)*100:.0f}%)')
print()
print(f'🔢 Características totales: {len(motor.caracteristicas)}')
print(f'📦 Secuencias LSTM (ventana={motor.config["preparacion"]["ventana_temporal"]}):')
print(f'   • Train: {motor.X_train_seq.shape}')
print(f'   • Test:  {motor.X_test_seq.shape}')

---
## 🤖 FASE 4: Modelado

### Entrenar los 5 algoritmos

In [ ]:
# Entrenar todos los algoritmos
print('=' * 60)
print('ENTRENANDO LOS 5 ALGORITMOS')
print('=' * 60)

motor.entrenar_todos()

print()
print(f'✅ Algoritmos entrenados exitosamente: {len(motor.modelos)}')
for nombre, info in motor.modelos.items():
    print(f'   • {nombre}: {info["tiempo_entrenamiento"]:.2f}s')

---
## 📈 FASE 5: Evaluación Comparativa

### Evaluar todos los algoritmos

In [ ]:
# Evaluar y comparar
print('=' * 60)
print('EVALUACIÓN COMPARATIVA')
print('=' * 60)

motor.evaluar_todos()
df_comparativa, puntuaciones, mejor = motor.comparar_algoritmos()

print()
print(f'🏆 MEJOR ALGORITMO: {mejor.upper()}')
print(f'   Puntuación general: {puntuaciones[mejor]["puntuacion_general"]:.4f}')

### Tabla Comparativa Completa

In [ ]:
# Mostrar tabla comparativa
df_final = motor.obtener_tabla_comparativa()
df_final

### Visualización Comparativa

In [ ]:
# Gráfico comparativo de métricas principales
resultados = list(motor.resultados_evaluacion.values())
metricas_df = pd.DataFrame([
    {'Algoritmo': r['algoritmo'].upper(), 
     'F1-Score': r['f1_score'],
     'AUC-ROC': r['auc_roc'],
     'Precisión': r['accuracy']}
    for r in resultados
])

fig = px.bar(metricas_df.melt(id_vars='Algoritmo', var_name='Métrica', value_name='Valor'),
             x='Algoritmo', y='Valor', color='Métrica', barmode='group',
             title='📊 Comparativa de Rendimiento por Algoritmo',
             color_discrete_map={'F1-Score': '#3498db', 'AUC-ROC': '#2ecc71', 'Precisión': '#e74c3c'},
             height=500)
fig.update_layout(yaxis_range=[0, 1.1])
fig.show()

In [ ]:
# Gráfico de tiempos
tiempos_df = pd.DataFrame([
    {'Algoritmo': r['algoritmo'].upper(), 
     'Tiempo Inferencia (ms)': r['tiempo_inferencia_ms']}
    for r in resultados
])

fig = px.bar(tiempos_df, x='Algoritmo', y='Tiempo Inferencia (ms)',
             title='⏱️ Tiempo de Inferencia por Predicción',
             color='Tiempo Inferencia (ms)', color_continuous_scale='Reds',
             height=450, log_y=True)
fig.show()

### Desglose de Puntuaciones

In [ ]:
# Radar chart de puntuaciones por criterio
categorias = ['Rendimiento', 'Tiempo', 'Interpretabilidad', 'Mantenibilidad']

fig = go.Figure()

for alg, punt in puntuaciones.items():
    fig.add_trace(go.Scatterpolar(
        r=[punt.get('rendimiento', 0), punt.get('tiempo', 0), 
           punt.get('interpretabilidad', 0), punt.get('mantenibilidad', 0)],
        theta=categorias,
        fill='toself',
        name=alg.upper()
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='🎯 Perfil de Puntuaciones por Criterio',
    height=600
)
fig.show()

### Importancia de Características (Random Forest)

In [ ]:
# Importancia de características del mejor algoritmo
info_rf = motor.modelos.get('random_forest', {})
importancias = info_rf.get('importancias', [])

if importancias:
    imp_df = pd.DataFrame(importancias[:15], columns=['Característica', 'Importancia'])
    imp_df['Característica'] = imp_df['Característica'].astype(str).str.replace('_', ' ').str.title()
    
    fig = px.bar(imp_df, x='Importancia', y='Característica', orientation='h',
                 title='📊 Top 15 Características Más Importantes (Random Forest)',
                 color='Importancia', color_continuous_scale='Reds',
                 height=600)
    fig.update_layout(yaxis={'categoryorder':'total ascending'})
    fig.show()
else:
    print('No hay datos de importancia de características')

---
## 🔮 FASE 6: Despliegue - Prueba de Predicción

In [ ]:
# Prueba de predicción con el mejor algoritmo
print('🔮 Prueba de Predicción:')
print('=' * 50)

# Usar una muestra del conjunto de prueba
muestra = motor.X_test[0]
resultado = motor.predecir(muestra)

for k, v in resultado.items():
    if k != 'factores_influyentes':
        print(f'  {k}: {v}')

print()
print('Factores influyentes:')
for factor, imp in resultado.get('factores_influyentes', [])[:5]:
    print(f'  • {str(factor).replace("_", " ").title()}: {imp*100:.1f}%')

---
## 💾 Guardar Modelos

In [ ]:
# Guardar modelos entrenados
motor.guardar()
print('✅ Modelos guardados exitosamente en disco')
print(f'   Directorio: {os.path.join(PROJECT_DIR, "models")}')

---
## 📋 Resumen de Conclusiones

In [ ]:
print('=' * 60)
print('CONCLUSIONES DEL ANÁLISIS')
print('=' * 60)
print()
print(f'🏆 Algoritmo seleccionado: {motor.mejor_algoritmo.upper()}')
print(f'📊 Puntuación general: {puntuaciones[motor.mejor_algoritmo]["puntuacion_general"]:.4f}')
print()
print('📈 Métricas del mejor algoritmo:')
mejor_metricas = motor.resultados_evaluacion[motor.mejor_algoritmo]
print(f'   • Accuracy:   {mejor_metricas["accuracy"]:.4f}')
print(f'   • F1-Score:   {mejor_metricas["f1_score"]:.4f}')
print(f'   • AUC-ROC:    {mejor_metricas["auc_roc"]:.4f}')
print(f'   • Inferencia: {mejor_metricas["tiempo_inferencia_ms"]:.2f} ms')
print()
print('💡 Razones de la selección:')
print('   1. Rendimiento predictivo superior o igual al mejor')
print('   2. Velocidad de inferencia adecuada para tiempo real')
print('   3. Alta interpretabilidad para ingenieros de mantenimiento')
print('   4. Facilidad de mantenimiento y reentrenamiento')
print('   5. Madurez tecnológica y soporte comunitario')

---
**Fin del Análisis** 🎉

El motor de IA está listo para ser integrado en la aplicación Streamlit.  
Ver la pestaña **⚙️ Motor IA** en la interfaz web para interactuar con los modelos.